> **복사본 (2026-10-06)** — 원본: `share_fast_model_v2_20261002/colab_fast_model_v2.ipynb`(팀원, 2026-10-02). 원본 셀은 고치지 않았고 아래 세 가지만 다르다.
> 1. 결과 폴더: `results/fast_model_v2` → `results/fast_model_v2_fullprec` (기존 결과와 섞이지 않게)
> 2. 1번 셀 바로 뒤: 기존 폴더의 게이트 값·DINOv2 특징을 새 폴더로 복사해 재사용 (있을 때만)
> 3. 9-1 셀 추가: 반올림 없는 확률 `predictions_fullprec.csv` 저장 + AUROC 확인
>
> 빠른 모델은 DINOv2 고정·증강 없음·로지스틱 회귀(결정적)라 시드를 바꿔도 결과가 같다. 그래서 반복 실행 없이 1회만 돌린다.


# 빠른 언더필 판정 모델 v2: 게이트 + 베드 마스킹 + DINOv2 고정 특징 + 로지스틱 회귀

**이 노트북만 Drive에 올리면 됩니다.** 게이트 코드는 2번 셀이 직접 만들고, 분할표는 Drive에 이미 있는 zip에서 자동으로 꺼냅니다.

**비교하는 4가지 조건** (같은 분할, 같은 모델)
| 조건 | 게이트 v2 | 베드 마스킹 |
| --- | --- | --- |
| A. 기준선 | 없음 | 없음 |
| B. 마스킹만 | 없음 | 있음 |
| C. 게이트만 | 있음 | 없음 |
| D. 게이트 + 마스킹 (제안) | 있음 | 있음 |

A↔B, C↔D를 비교하면 마스킹 효과를, A↔C를 비교하면 게이트 효과를 따로 볼 수 있습니다.

**구성:** 고정 크롭 [0, 250, 1000, 1250] → (마스킹) → 518×518 → DINOv2 ViT-B/14 고정 특징 1,536개 → 표준화 + 로지스틱 회귀(C=1.0), 확률 0.5 이상이면 언더필. 평가는 시간 블록 5-fold(두현님 분할표와 같음).

**Drive `산공종설/`에 필요한 것:** `underfill_originals_4000.zip`, 그리고 분할표가 든 파일 하나 (`folds_all4000.csv`, `stfpm_compare_bundle_v4.zip`, 두현님 `share_pipeline_3models_20261002.zip` 중 아무거나)

**실행:** 런타임 > 런타임 유형 변경 > GPU → 1번부터 차례로. 오래 걸리는 건 6번(특징 추출, 마스킹 전·후 두 벌) 한 번뿐입니다. 결과는 `산공종설/results/fast_model_v2/`에 저장되고, 끊겨도 저장된 값은 다시 계산하지 않습니다.

## 1. GPU 확인 + Drive 연결 + 파일 찾기

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob, sys, zipfile, shutil
DRIVE_DIR = "/content/drive/MyDrive/산공종설"   # 폴더 이름 다르면 여기만 바꾸기
OUT = f"{DRIVE_DIR}/results/fast_model_v2_fullprec"   # 복사본: 기존 결과 폴더와 분리
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(f"{DRIVE_DIR}/underfill_originals_4000.zip"), "underfill_originals_4000.zip 없음"

# 분할표: Drive에 있는 것 중 하나에서 꺼냄 (네 파일 모두 같은 분할)
FOLDS = "/content/folds_all4000.csv"
cands = [(f"{DRIVE_DIR}/folds_all4000.csv", None),
         (f"{DRIVE_DIR}/stfpm_compare_bundle_v4.zip", "folds_all4000.csv"),
         (f"{DRIVE_DIR}/share_pipeline_3models_20261002.zip", "code/splits/folds.csv"),
         (f"{DRIVE_DIR}/TalkFile_share_pipeline_3models_20261002.zip", "code/splits/folds.csv")]
for path, inner in cands:
    if not os.path.exists(path): continue
    if inner is None:
        shutil.copy(path, FOLDS)
    else:
        with zipfile.ZipFile(path) as z:
            n = next(x for x in z.namelist() if x.endswith(inner))
            with z.open(n) as src, open(FOLDS, "wb") as dst: shutil.copyfileobj(src, dst)
    print("분할표:", path); break
assert os.path.exists(FOLDS), "분할표를 찾지 못함: 위 파일 중 하나를 산공종설/에 올려 주세요"

In [ ]:
# 기존 실행의 게이트 값·DINOv2 특징이 있으면 새 폴더로 복사해서 재사용 (특징 추출 수십 분 생략)
# 원본 폴더 파일은 읽기만 하고 바꾸지 않는다
PREV = f"{DRIVE_DIR}/results/fast_model_v2"
for name in ("gate_values_v2.csv", "feats_nomask.npy", "feats_mask.npy"):
    a, b = f"{PREV}/{name}", f"{OUT}/{name}"
    if os.path.exists(a) and not os.path.exists(b):
        shutil.copy(a, b); print("재사용:", name)
    print("있음" if os.path.exists(b) else "없음 → 아래 셀에서 새로 계산", name)


## 2. 게이트 + 마스킹 코드 (filament_gate.py v2)

In [ ]:
%%writefile /content/filament_gate.py
"""질감 기준 필라멘트 게이트 + 베드 마스킹 v2 (2026-10-02, 팀원)

하는 일 (고정 크롭 1000x1000 하나에 대해)
  1. 출력물 영역 찾기 -> 면적 비율(ratio)
  2. 게이트: ratio < 0.1 이면 판정 보류 (베드만 찍힌 사진)
  3. 마스킹: 출력물 바깥(베드)을 회색으로 칠한 크롭을 돌려줌 -> 모델 입력용
     (모델이 '반짝이 = 언더필'로 판단하는 문제를 막기 위함)

출력물 영역 찾는 순서 (250x250으로 줄여서 계산)
  a. 픽셀마다 주변 9x9 밝기 표준편차(질감)와 평균 밝기 계산
  b. 표준편차 < 10 (매끈) & 밝기 > 110 -> 출력물 후보
     (v1의 채도 조건은 삭제: 360장 시험에서 빼도 결과 동일, 자동으로 정하면 흰 출력물끼리 갈라버림)
  c. 7x7 열기·닫기, 화면 2% 미만 덩어리 버림
  d. 흔들린 베드 얼룩 제거: 덩어리의 (표준편차 중앙값 >= 5) & (밝기 중앙값 < 155) & (볼록도 < 0.8)
     -> 셋 다 해당하면 버림. 번진 반짝이는 덜 매끈하고 어둡고 모양이 들쭉날쭉함
  e. 남은 덩어리마다 바깥 윤곽 안쪽을 전부 채움 -> 언더필의 틈(결함 증거)이 회색으로 지워지지 않게
  f. 마스킹 때는 윤곽을 2픽셀(원본 기준 약 8픽셀) 넓혀서 테두리가 잘리지 않게 함

사용법
  from filament_gate import crop_roi, filament_ratio, gate, mask_bed
  crop = crop_roi(img)                     # 원본 1600x1200 -> 고정 크롭 1000x1000
  r = filament_ratio(crop)                 # RGB 기본 (두현님 pipeline.py와 같은 형태)
  ok, r = gate(crop, bgr=True)             # cv2로 읽은 BGR이면 bgr=True. ok=False면 판정 보류
  masked, r = mask_bed(crop, bgr=True)     # 베드를 회색 칠한 크롭 (입력과 같은 크기·색 순서)

필요 패키지: numpy, opencv-python (Colab 기본 설치됨)
"""
import cv2
import numpy as np

VERSION = "v2-mask"
ROI = (0, 250, 1000, 1250)       # ymin, xmin, ymax, xmax
THRESHOLD = 0.1                  # 출력물 면적 이 비율 미만이면 판정 보류
SIZE, WIN = 250, 9               # 계산용 축소 크기, 질감 보는 창
SD_T, MEAN_T = 10, 110           # 출력물 후보: 매끈 & 밝음
MIN_BLOB = 0.02                  # 화면 2% 미만 덩어리 버림
BLUR_SD, BLUR_MEAN, BLUR_SOL = 5, 155, 0.8   # 흔들린 베드 얼룩 판정 (셋 다 해당하면 버림)
MARGIN = 2                       # 마스킹 여유 (축소 기준 픽셀)
GRAY_RGB = (124, 116, 104)       # ImageNet 평균색 -> 정규화하면 거의 0
_K7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
_KM = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * MARGIN + 1, 2 * MARGIN + 1))


def crop_roi(img):
    """원본 프레임 -> 고정 크롭. 색 순서 상관없음."""
    y0, x0, y1, x1 = ROI
    assert img.shape[0] >= y1 and img.shape[1] >= x1, f"원본 크기가 크롭보다 작음: {img.shape}"
    return img[y0:y1, x0:x1]


def object_mask(crop, bgr=False):
    """출력물 영역 마스크 (SIZE x SIZE, 0/1, 윤곽 안 채움)."""
    crop = np.ascontiguousarray(crop, dtype=np.uint8)
    small = cv2.resize(crop, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY if bgr else cv2.COLOR_RGB2GRAY).astype(np.float32)
    m = cv2.blur(g, (WIN, WIN))
    sd = np.sqrt(np.maximum(cv2.blur(g * g, (WIN, WIN)) - m * m, 0))
    raw = ((sd < SD_T) & (m > MEAN_T)).astype(np.uint8)
    raw = cv2.morphologyEx(cv2.morphologyEx(raw, cv2.MORPH_OPEN, _K7), cv2.MORPH_CLOSE, _K7)
    n, lab, st, _ = cv2.connectedComponentsWithStats(raw)
    out = np.zeros_like(raw)
    for j in range(1, n):
        if st[j, cv2.CC_STAT_AREA] < MIN_BLOB * SIZE * SIZE:
            continue
        comp = lab == j
        cs, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cs, key=cv2.contourArea)
        sol = cv2.contourArea(c) / max(cv2.contourArea(cv2.convexHull(c)), 1.0)
        if np.median(sd[comp]) >= BLUR_SD and np.median(m[comp]) < BLUR_MEAN and sol < BLUR_SOL:
            continue                                    # 흔들린 베드 얼룩
        cv2.drawContours(out, cs, -1, 1, thickness=-1)  # 바깥 윤곽 안쪽 전부 채움 (틈 유지)
    return out


def filament_ratio(crop, bgr=False):
    """출력물 면적 비율 (0~1). 두현님 filament_ratio(rgb_crop)와 같은 형태."""
    return float(object_mask(crop, bgr).mean())


def gate(crop, threshold=THRESHOLD, bgr=False):
    """(통과 여부, 비율). 통과 못 하면 판정 보류."""
    r = filament_ratio(crop, bgr)
    return r >= threshold, r


def mask_bed(crop, bgr=False):
    """베드(출력물 바깥)를 회색으로 칠한 크롭과 비율. 크기·색 순서는 입력과 같음."""
    crop = np.ascontiguousarray(crop, dtype=np.uint8)
    om = object_mask(crop, bgr)
    keep = cv2.dilate(om, _KM)
    keep = cv2.resize(keep, (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    out = crop.copy()
    out[~keep] = GRAY_RGB[::-1] if bgr else GRAY_RGB
    return out, float(om.mean())


def overlay(crop, bgr=False, color=(255, 40, 40), alpha=0.55):
    """확인용: 출력물로 판단한 영역을 빨간색으로 칠한 RGB 이미지 (SIZE x SIZE)."""
    om = object_mask(crop, bgr)
    small = cv2.resize(np.ascontiguousarray(crop, dtype=np.uint8), (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    rgb = small[:, :, ::-1].copy() if bgr else small.copy()
    k = om == 1
    rgb[k] = ((1 - alpha) * rgb[k] + alpha * np.array(color)).astype(np.uint8)
    return rgb


def otsu_ratio(crop, bgr=False):
    """비교용: 기존 게이트 값 (원욱님 check_filament_presence와 같은 Otsu 밝은 픽셀 비율)."""
    g = cv2.cvtColor(np.ascontiguousarray(crop, dtype=np.uint8), cv2.COLOR_BGR2GRAY if bgr else cv2.COLOR_RGB2GRAY)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return float((b > 0).mean())


def _self_check():
    flat = np.full((1000, 1000, 3), 200, np.uint8)
    rng = np.random.default_rng(0)
    glitter = (rng.random((1000, 1000, 3)) > 0.7).astype(np.uint8) * 230
    assert filament_ratio(flat) > 0.95, filament_ratio(flat)
    assert filament_ratio(glitter) < 0.05, filament_ratio(glitter)
    half = glitter.copy(); half[500:] = 200
    mk, r = mask_bed(half)
    assert 0.4 < r < 0.6 and (mk[:400] == GRAY_RGB).all() and (mk[600:] == 200).all()


if __name__ == "__main__":
    # python filament_gate.py 사진.jpg ... -> 사진별 비율, 통과/보류, 기존 Otsu 값
    import sys
    _self_check()
    for p in sys.argv[1:]:
        img = cv2.imread(p)
        ok, r = gate(crop_roi(img), bgr=True)
        print(f"{p}\tratio={r:.3f}\t{'통과' if ok else '보류'}\t(otsu={otsu_ratio(crop_roi(img), bgr=True):.3f})")
    if len(sys.argv) == 1:
        print("self-check ok", VERSION)

## 3. 원본 풀기 + 게이트 불러오기
마지막 줄에 `4000`과 `v2-mask`가 나와야 합니다.

In [ ]:
import importlib
if len(glob.glob("/content/orig/**/*.jpg", recursive=True)) < 4000:
    !cp "{DRIVE_DIR}/underfill_originals_4000.zip" /content/
    !unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
sys.path.insert(0, "/content")
import filament_gate as fg
importlib.reload(fg); fg._self_check()
print(len(glob.glob("/content/orig/**/*.jpg", recursive=True)), fg.VERSION)

## 4. 설정 + 사진 목록

In [ ]:
import numpy as np, pandas as pd, cv2, time
from pathlib import Path
GATE_T, PROB_T, IMG = fg.THRESHOLD, 0.5, 518
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
folds = pd.read_csv(FOLDS, encoding="utf-8-sig")
folds["source"] = folds.source.str.replace("\\", "/", regex=False)
imgs = (folds[folds.role == "val"][["source", "label", "label_id", "frame_index", "fold"]]
        .rename(columns={"fold": "test_fold"}).sort_values("source").reset_index(drop=True))
src_idx = {s: i for i, s in enumerate(imgs.source)}
print(len(imgs), imgs.label.value_counts().to_dict())

## 5. 게이트 값 (1~2분, 한 번만)
게이트 확인 노트북(v2)을 이미 돌렸다면 그 결과를 가져옵니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
gate_file = f"{OUT}/gate_values_v2.csv"
other = f"{DRIVE_DIR}/results/gate_check_v2/gate_values_v2.csv"
if not os.path.exists(gate_file) and os.path.exists(other):
    shutil.copy(other, gate_file); print("게이트 확인 노트북 결과 재사용")
if os.path.exists(gate_file):
    gv = pd.read_csv(gate_file, encoding="utf-8-sig")
else:
    t0 = time.time()
    def one(s): return fg.filament_ratio(fg.crop_roi(cv2.imread(f"/content/orig/{s}")), bgr=True)
    with ThreadPoolExecutor(4) as ex: vals = list(ex.map(one, imgs.source))
    gv = pd.DataFrame({"source": imgs.source, "new": np.round(vals, 4)})
    gv.to_csv(gate_file, index=False, encoding="utf-8-sig"); print(f"완료: {time.time()-t0:.0f}초")
imgs = imgs.drop(columns=["new"], errors="ignore").merge(gv[["source", "new"]], on="source", how="left")
assert imgs.new.notna().all()
imgs["passed"] = imgs.new >= GATE_T
print("게이트 보류:", (~imgs.passed).groupby(imgs.label).sum().to_dict(), "| 통과", int(imgs.passed.sum()), "장")

## 6. DINOv2 특징 추출: 마스킹 전·후 두 벌 (한 번만)
사진마다 원본 크롭과 마스킹 크롭을 둘 다 넣어 `feats_nomask.npy`, `feats_mask.npy`로 저장합니다. 이미 있으면 불러오기만 합니다. 처음 실행 때 DINOv2 가중치를 받습니다(HF_TOKEN 경고는 무시). 걸린 시간이 마지막에 출력됩니다.

In [ ]:
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.transforms.v2 import functional as TF

def to_input(crop_rgb):
    # RGB uint8 크롭 -> 518 축소 -> 0~1 -> ImageNet 정규화
    t = torch.from_numpy(np.ascontiguousarray(crop_rgb)).permute(2, 0, 1)
    t = TF.resize(t, [IMG, IMG], interpolation=v2.InterpolationMode.BILINEAR, antialias=True)
    return TF.normalize(t.float() / 255.0, MEAN, STD)

def both_inputs(bgr):
    crop = fg.crop_roi(bgr)
    masked, _ = fg.mask_bed(crop, bgr=True)
    return to_input(crop[:, :, ::-1]), to_input(masked[:, :, ::-1])

dev = "cuda"
dn = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).eval().to(dev)
NPT = dn.num_prefix_tokens

@torch.no_grad()
def extract(x):
    with torch.autocast("cuda", dtype=torch.float16):
        tok = dn.forward_features(x.to(dev)).float()
    return torch.cat([tok[:, 0], tok[:, NPT:].mean(1)], 1).cpu()   # CLS + 조각 평균

ff = {"nomask": f"{OUT}/feats_nomask.npy", "mask": f"{OUT}/feats_mask.npy"}
if all(os.path.exists(f) for f in ff.values()):
    X = {k: np.load(f) for k, f in ff.items()}; print("저장된 특징 불러옴")
else:
    class Imgs(Dataset):
        def __len__(self): return len(imgs)
        def __getitem__(self, i): return both_inputs(cv2.imread(f"/content/orig/{imgs.source[i]}"))
    out = {"nomask": [], "mask": []}; t0 = time.time()
    for b, (x0, x1) in enumerate(DataLoader(Imgs(), batch_size=16, num_workers=2)):
        out["nomask"].append(extract(x0)); out["mask"].append(extract(x1))
        if b % 50 == 0: print(f"{(b+1)*16:5d} / {len(imgs)}장 ({time.time()-t0:.0f}초)")
    X = {k: torch.cat(v).numpy() for k, v in out.items()}
    for k in X: np.save(ff[k], X[k])
    print(f"완료: {time.time()-t0:.0f}초")
print({k: v.shape for k, v in X.items()})

## 7. 4가지 조건 비교 (몇 초)
지표는 **판정한 사진 기준**입니다. 게이트가 있는 조건은 `판정비율`을 같이 보세요.

참고(같은 분할, 두현님 2026-10-02, 기존 게이트로 3,951장, 시드 3개 평균): ConvNeXt-Tiny 정확도 0.931, AUROC 0.991, 오탐 219, 미탐 52 / ResNet18 0.922, 0.973, 237, 71

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def fit(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    return sc, LogisticRegression(max_iter=5000).fit(sc.transform(Xtr), ytr)

def cv_prob(feat, use_gate):
    p = np.full(len(imgs), np.nan); Xf = X[feat]; y = imgs.label_id.values
    for k in range(5):
        f = folds[folds.fold == k]
        itr, ite = f[f.role == "train"].source.map(src_idx).values, f[f.role == "val"].source.map(src_idx).values
        if use_gate:
            itr, ite = itr[imgs.passed.values[itr]], ite[imgs.passed.values[ite]]
        sc, clf = fit(Xf[itr], y[itr])
        p[ite] = clf.predict_proba(sc.transform(Xf[ite]))[:, 1]
    return p

def summarize(p):
    y = imgs.label_id.values; ok = ~np.isnan(p); pred = (p >= PROB_T).astype(int)
    tp = ((pred == 1) & (y == 1) & ok).sum(); fn = ((pred == 0) & (y == 1) & ok).sum()
    fp = ((pred == 1) & (y == 0) & ok).sum(); tn = ((pred == 0) & (y == 0) & ok).sum()
    acc_k = [((pred == y) & ok & (imgs.test_fold == k)).sum() / (ok & (imgs.test_fold == k)).sum() for k in range(5)]
    return {"판정비율": round(ok.mean(), 4), "AUROC": round(roc_auc_score(y[ok], p[ok]), 4),
            "정확도": round((tp + tn) / ok.sum(), 4), "언더필 재현율": round(tp / (tp + fn), 4),
            "정상 재현율": round(tn / (tn + fp), 4), "오탐": int(fp), "미탐": int(fn),
            "fold별 정확도": " ".join(f"{a:.3f}" for a in acc_k)}

CONDS = {"A 기준선": ("nomask", False), "B 마스킹만": ("mask", False),
         "C 게이트만": ("nomask", True), "D 게이트+마스킹": ("mask", True)}
prob = {k: cv_prob(*v) for k, v in CONDS.items()}
summary = pd.DataFrame([{"조건": k, **summarize(p)} for k, p in prob.items()])
summary.to_csv(f"{OUT}/summary.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.width", 220)
print(summary.to_string(index=False))

## 8. 마스킹이 어디서 효과가 있었나
게이트를 통과한 사진만 놓고, 출력물 비율 구간별로 C(마스킹 없음)와 D(마스킹)의 오탐·미탐을 비교합니다. 예상: 베드가 많이 보이는 구간(0.1~0.4)에서 **오탐이 줄고**, 미탐은 비슷하거나 조금 늘 수 있음(반짝이 단서가 사라지므로).

In [ ]:
y = imgs.label_id.values
bins = pd.cut(imgs.new, [0.1 - 1e-9, 0.2, 0.4, 0.7, 1.01], labels=["0.1~0.2", "0.2~0.4", "0.4~0.7", "0.7~1.0"])
rows = []
for name in ["C 게이트만", "D 게이트+마스킹"]:
    p = prob[name]; ok = ~np.isnan(p); pred = (p >= PROB_T).astype(int)
    for b in bins.cat.categories:
        s = ok & (bins == b).values
        rows.append({"조건": name, "출력물 비율": b, "정상 수": int((s & (y == 0)).sum()), "오탐": int((s & (y == 0) & (pred == 1)).sum()),
                     "언더필 수": int((s & (y == 1)).sum()), "미탐": int((s & (y == 1) & (pred == 0)).sum())})
bt = pd.DataFrame(rows).pivot(index="출력물 비율", columns="조건", values=["오탐", "미탐"])
bt.insert(0, "정상 수", pd.DataFrame(rows).groupby("출력물 비율")["정상 수"].first())
bt.insert(1, "언더필 수", pd.DataFrame(rows).groupby("출력물 비율")["언더필 수"].first())
bt.to_csv(f"{OUT}/by_ratio_bin.csv", encoding="utf-8-sig"); print(bt)

## 9. 사진별 결과 CSV + 미탐·오탐 썸네일 (D 조건)
- `predictions.csv`: 사진 한 줄씩, 4가지 조건의 확률과 결과(정답/미탐/오탐/보류). 엑셀에서 열림
- `thumbs/FN_p01.png`, `thumbs/FP_p01.png`…: D 조건에서 틀린 사진. **모델이 실제로 본 마스킹 입력**으로 보여 줍니다

In [ ]:
import matplotlib.pyplot as plt
def result(p):
    pred = (p >= PROB_T).astype(int)
    return np.where(np.isnan(p), "보류", np.where(pred == y, "정답", np.where(y == 1, "미탐", "오탐")))
df = pd.DataFrame({"사진": imgs.source, "정답": imgs.label, "시험fold": imgs.test_fold, "프레임": imgs.frame_index, "출력물비율": imgs.new})
for k, p in prob.items():
    df[f"확률_{k[0]}"] = np.round(p, 4); df[f"결과_{k[0]}"] = result(p)
df.to_csv(f"{OUT}/predictions.csv", index=False, encoding="utf-8-sig")
print(df["결과_D"].value_counts().to_dict())

TD = f"{OUT}/thumbs"; os.makedirs(TD, exist_ok=True)
pD, rD = prob["D 게이트+마스킹"], result(prob["D 게이트+마스킹"])
for kind, tag in (("미탐", "FN"), ("오탐", "FP")):
    idx = np.where(rD == kind)[0]; idx = idx[np.argsort(imgs.frame_index.values[idx])]
    for pg in range(0, len(idx), 24):
        chunk = idx[pg:pg + 24]
        fig, axs = plt.subplots(4, 6, figsize=(15, 11))
        for a in axs.flat: a.axis("off")
        for a, i in zip(axs.flat, chunk):
            mk, _ = fg.mask_bed(fg.crop_roi(cv2.imread(f"/content/orig/{imgs.source[i]}")), bgr=True)
            a.imshow(cv2.resize(mk, (250, 250), interpolation=cv2.INTER_AREA)[:, :, ::-1])
            a.set_title(f"{imgs.label[i][:9]} #{imgs.frame_index[i]}  p={pD[i]:.2f}\nratio={imgs.new[i]:.2f}", fontsize=8)
        fig.suptitle(f"D {tag} {pg+1}-{pg+len(chunk)} / {len(idx)}")
        plt.tight_layout(); plt.savefig(f"{TD}/{tag}_p{pg//24+1:02d}.png", dpi=80)
        if pg == 0: plt.show()
        plt.close(fig)
    print(tag, len(idx), "장")

## 9-1. 반올림 없는 확률 저장 (추가한 셀)
위 `predictions.csv`는 확률을 소수 넷째 자리로 반올림해 저장한다. 확률이 0이나 1 근처에 몰려 있어서 반올림하면 동점이 많이 생기고, AUROC가 실제보다 낮게 계산된다(ConvNeXt 확률로 시험해 보니 0.9954 → 0.9921). 다른 모델과 AUROC를 공정하게 비교하려고 반올림 없는 확률을 따로 저장한다.

마지막 표에서 `AUROC_전체정밀도`가 7번 셀 summary의 AUROC와 같아야 한다(같으면 저장이 맞게 된 것).


In [ ]:
full = pd.DataFrame({"사진": imgs.source, "정답": imgs.label, "시험fold": imgs.test_fold, "프레임": imgs.frame_index, "출력물비율": imgs.new})
for k, p in prob.items():
    full[f"확률_{k[0]}"] = p
full.to_csv(f"{OUT}/predictions_fullprec.csv", index=False, encoding="utf-8-sig", float_format="%.10g")

check = []
for k, p in prob.items():
    ok = ~np.isnan(p)
    check.append({"조건": k, "판정 수": int(ok.sum()),
                  "AUROC_전체정밀도": round(roc_auc_score(y[ok], p[ok]), 6),
                  "AUROC_4자리반올림": round(roc_auc_score(y[ok], np.round(p[ok], 4)), 6)})
print(pd.DataFrame(check).to_string(index=False))
print("저장:", f"{OUT}/predictions_fullprec.csv")


## 10. 최종 모델 저장 (D 조건)
게이트를 통과한 전체 사진의 마스킹 특징으로 한 번 학습해 `fast_model_v2.joblib`로 저장합니다. 평가 점수는 7번(5-fold)을 기준으로 보세요.

In [ ]:
import joblib
ia = np.where(imgs.passed.values)[0]
t0 = time.time(); sc, clf = fit(X["mask"][ia], y[ia])
bundle = {"scaler": sc, "clf": clf, "backbone": "vit_base_patch14_dinov2.lvd142m", "img": IMG, "roi": fg.ROI,
          "gate_version": fg.VERSION, "gate_threshold": GATE_T, "mask": True, "prob_threshold": PROB_T, "n_train": len(ia)}
joblib.dump(bundle, f"{OUT}/fast_model_v2.joblib")
print(f"학습 {len(ia)}장, {time.time()-t0:.1f}초 → {OUT}/fast_model_v2.joblib")

## 11. 새 사진 판정 + 속도 측정
`judge(경로)`: 게이트 → 마스킹 → 특징 → 확률 → 판정. `JUDGE_DIR`에 폴더를 넣으면 폴더 전체를 판정해 `judged.csv`로 저장합니다.

아래 예시는 동작·속도 확인용 원본 20장입니다. **10번 최종 모델의 학습에 들어간 사진이라 정확도 확인용이 아닙니다.** 속도는 사진 읽기 포함 한 장 처리 시간(첫 장 제외 중앙값, GPU)입니다.

In [ ]:
def judge(path, model=bundle):
    t0 = time.perf_counter()
    crop = fg.crop_roi(cv2.imread(str(path)))
    masked, r = fg.mask_bed(crop, bgr=True)
    prob_u = None
    if r >= model["gate_threshold"]:
        f = extract(to_input(masked[:, :, ::-1]).unsqueeze(0)).numpy()
        prob_u = float(model["clf"].predict_proba(model["scaler"].transform(f))[0, 1])
    torch.cuda.synchronize()
    verdict = "보류" if prob_u is None else ("언더필" if prob_u >= model["prob_threshold"] else "정상")
    return {"사진": Path(path).name, "출력물비율": round(r, 3), "확률": None if prob_u is None else round(prob_u, 4),
            "판정": verdict, "ms": round((time.perf_counter() - t0) * 1000, 1)}

JUDGE_DIR = None   # 예: f"{DRIVE_DIR}/new_frames"
files = sorted(glob.glob(f"{JUDGE_DIR}/*.jpg")) if JUDGE_DIR else [f"/content/orig/{s}" for s in imgs.source.iloc[::200]]
res = pd.DataFrame([judge(f) for f in files])
res.to_csv(f"{OUT}/judged.csv", index=False, encoding="utf-8-sig")
print(res.head(10).to_string(index=False))
print(f"\n{len(res)}장 | 한 장 처리 중앙값 {res.ms.iloc[1:].median():.0f}ms (최대 {res.ms.iloc[1:].max():.0f}ms, GPU)")

## 12. 런타임 반납

In [ ]:
from google.colab import runtime
runtime.unassign()